In [1]:
from google.colab import files

uploaded = files.upload()

Saving time_series_60min_singleindex.csv to time_series_60min_singleindex.csv


In [29]:
import pandas as pd
import numpy as np

df = pd.read_csv('time_series_60min_singleindex.csv')

energy = pd.DataFrame()

energy['date'] = pd.to_datetime(df['utc_timestamp'])
energy['price'] = df['DE_LU_price_day_ahead']

energy = energy.dropna()
energy = energy.set_index('date')

energy['return'] = energy['price'].diff()

energy = energy.dropna()

print(energy.shape)

energy.head()

(17539, 2)


,price,return
date,,
2018-10-01 00:00:00+00:00,51.41,-4.69
2018-10-01 01:00:00+00:00,47.38,-4.03
2018-10-01 02:00:00+00:00,47.59,0.21
2018-10-01 03:00:00+00:00,51.61,4.02
2018-10-01 04:00:00+00:00,69.13,17.52


In [30]:
#descriptive  of risk factor
print(
    energy['return'].describe()
)

from scipy.stats import skew, kurtosis

print(
    "Skew:",
    skew(energy['return'])
)

print(
    "Kurtosis:",
    kurtosis(energy['return'])
)

count    17539.000000
mean        -0.001237
std          5.551050
min       -101.910000
25%         -2.445000
50%         -0.320000
75%          2.095000
max        109.230000
Name: return, dtype: float64
Skew: 0.48566319834168453
Kurtosis: 26.21171337626122


1. Μέση απόδοση ≈ 0

Αυτό είναι αναμενόμενο:

E(return) ≈ 0

Οι αγορές ηλεκτρικής ενέργειας δεν έχουν μακροχρόνιο drift σε ωριακό επίπεδο.

2. Πολύ υψηλή kurtosis
Kurtosis = 26.2

Αυτό σημαίνει:

ακραία γεγονότα,
price spikes,
fat tails.

Άρα η κανονική κατανομή θα υποεκτιμήσει σημαντικά τον κίνδυνο.

3. Ασυμμετρία
Skew = 0.49

Υπάρχει ελαφριά δεξιά ουρά, δηλαδή μεγάλα θετικά price shocks.

Τι θα κάνουμε τώρα;

Θα κατασκευάσουμε τρία διαφορετικά risk measures:

1. Historical VaR

Τι έχω χάσει ιστορικά;

2. Parametric Gaussian VaR

Τι προβλέπει η κανονική κατανομή;

3. Expected Shortfall (CVaR)

Αν ξεπεράσω το VaR, πόσο θα χάσω κατά μέσο όρο;

In [31]:
#Historical VaR
var95 = np.percentile(
    energy['return'],
    5
)

var99 = np.percentile(
    energy['return'],
    1
)

print("Historical VaR 95%:", var95)
print("Historical VaR 99%:", var99)

Historical VaR 95%: -7.460000000000001
Historical VaR 99%: -14.224800000000004


In [32]:
#Expected Shortfall (CVaR)
cvar95 = energy.loc[
    energy['return'] <= var95,
    'return'
].mean()

cvar99 = energy.loc[
    energy['return'] <= var99,
    'return'
].mean()

print("CVaR 95%:", cvar95)
print("CVaR 99%:", cvar99)

CVaR 95%: -12.040045454545455
CVaR 99%: -21.022159090909092


In [33]:
#Parametric Gaussian VaR
from scipy.stats import norm

mu = energy['return'].mean()
sigma = energy['return'].std()

gaussian_var95 = norm.ppf(
    0.05,
    mu,
    sigma
)

gaussian_var99 = norm.ppf(
    0.01,
    mu,
    sigma
)

print(
    "Gaussian VaR95:",
    gaussian_var95
)

print(
    "Gaussian VaR99:",
    gaussian_var99
)

Gaussian VaR95: -9.131901862384503
Gaussian VaR99: -12.914910458680673


| Measure             | Value  |
| ------------------- | ------ |
| Historical VaR 95%  | -7.46  |
| Historical VaR 99%  | -14.22 |
| Historical CVaR 95% | -12.04 |
| Historical CVaR 99% | -21.02 |
| Gaussian VaR 95%    | -9.13  |
| Gaussian VaR 99%    | -12.91 |
VaR 95%
Historical : -7.46
Gaussian  : -9.13

Εδώ η Gaussian είναι λίγο πιο συντηρητική.

VaR 99%
Historical : -14.22
Gaussian  : -12.91

Εδώ συμβαίνει το αναμενόμενο:

Η κανονική κατανομή αρχίζει να υποεκτιμά τον ακραίο κίνδυνο.

CVaR

Το πιο ενδιαφέρον αποτέλεσμα:

VaR99  = -14.22
CVaR99 = -21.02

Αυτό σημαίνει ότι:

αν ξεπεράσουμε το 99% VaR, η πραγματική ζημιά είναι κατά μέσο όρο περίπου 50% μεγαλύτερη.

Αυτό είναι κλασικό χαρακτηριστικό των electricity markets λόγω:

price spikes,
supply shortages,
renewable intermittency.

In [34]:
#Monte Carlo Risk Engine
#Monte Carlo simulation
n_sim = 10000

mu = energy['return'].mean()
sigma = energy['return'].std()

mc_returns = np.random.normal(
    mu,
    sigma,
    n_sim
)

print(mc_returns[:10])

[  3.40615774   0.69877338  -0.118501    -4.76855887  -0.06413237
  -6.26962233 -11.122966    -0.79210269  -0.48812306   6.06580892]


In [35]:
#Monte Carlo VaR
mc_var95 = np.percentile(
    mc_returns,
    5
)

mc_var99 = np.percentile(
    mc_returns,
    1
)

print(
    "MC VaR95:",
    mc_var95
)

print(
    "MC VaR99:",
    mc_var99
)

MC VaR95: -9.173411087434015
MC VaR99: -13.131568229218034


In [36]:
#Monte Carlo CVaR
mc_cvar95 = mc_returns[
    mc_returns <= mc_var95
].mean()

mc_cvar99 = mc_returns[
    mc_returns <= mc_var99
].mean()

print(
    "MC CVaR95:",
    mc_cvar95
)

print(
    "MC CVaR99:",
    mc_cvar99
)

MC CVaR95: -11.598409738747899
MC CVaR99: -15.124881087640361


Σύγκριση όλων των risk models
| Method      | VaR95 |  VaR99 | CVaR95 | CVaR99 |
| ----------- | ----: | -----: | -----: | -----: |
| Historical  | -7.46 | -14.22 | -12.04 | -21.02 |
| Gaussian    | -9.13 | -12.91 |      — |      — |
| Monte Carlo | -9.17 | -13.13 | -11.60 | -15.12 |
1. Στο 95%
Historical VaR95 = -7.46
MC VaR95         = -9.17
Gaussian VaR95  = -9.13

Τα τρία μοντέλα είναι σχετικά κοντά.

2. Στο 99%
Historical VaR99 = -14.22
MC VaR99         = -13.13
Gaussian VaR99   = -12.91

Εδώ αρχίζουν να εμφανίζονται οι fat tails.

3. Το πιο σημαντικό εύρημα
Historical CVaR99 = -21.02
Monte Carlo CVaR99 = -15.12

Αυτό είναι ακριβώς το πρόβλημα των Gaussian risk models:

υποεκτιμούν τις ακραίες ζημιές στην αγορά ηλεκτρικής ενέργειας.
The German electricity market exhibits substantial tail risk. Historical simulation captures extreme events more effectively than Gaussian Monte Carlo approaches, suggesting that traditional risk models underestimate downside exposure in electricity trading portfolios.

In [37]:
#Stress Testing
#Scenario1
sigma = energy['return'].std()

stress_2sigma = -2*sigma
stress_3sigma = -3*sigma
stress_5sigma = -5*sigma

print("2σ stress:", stress_2sigma)
print("3σ stress:", stress_3sigma)
print("5σ stress:", stress_5sigma)

2σ stress: -11.102099870844794
3σ stress: -16.65314980626719
5σ stress: -27.755249677111987


Υπολογίζουμε θεωρητικά γεγονότα:

95% stress
99.7% stress
extreme black swan stress

In [38]:
#Scenario 2 — Historical worst case
print(
    "Worst historical loss:",
    energy['return'].min()
)

print(
    "Best historical gain:",
    energy['return'].max()
)

Worst historical loss: -101.91
Best historical gain: 109.22999999999999


Το 5σ event
5σ loss = -27.76

είναι ήδη μεγαλύτερο από το CVaR99:

CVaR99 = -21.02

δηλαδή ένα ακραίο stress event προκαλεί σημαντικά μεγαλύτερες απώλειες από αυτές που περιγράφει το 99% confidence interval.
Worst historical loss = -101.91

Αυτό είναι σχεδόν:

9 φορές μεγαλύτερο

από το VaR95 και περίπου:

5 φορές μεγαλύτερο

από το CVaR99.

Αυτό είναι η κλασική απόδειξη ότι:

οι αγορές ηλεκτρικής ενέργειας παρουσιάζουν extreme tail risk και τα παραδοσιακά Gaussian risk models υποεκτιμούν σημαντικά τον πραγματικό κίνδυνο.